# Process Handoff and DistributedDataParallel (Offline)

This advanced tutorial covers two delivered execution boundaries: handing a CPU Workflow artifact to another local process through TensorDict-native shared storage, and running rank-local TDHook operations around a caller-owned `DistributedDataParallel` model. TDHook does not launch workers, initialize multi-rank jobs, or aggregate results.

## Scope

Local shared-memory handoff preserves an already allocated TensorDict across a process boundary. DDP execution keeps the model, session, hooks, plans, and results local to each rank. Disk-backed memory-mapped caches and cross-rank artifact exchange are separate contracts and are not covered here.

In [ ]:
import tempfile
from pathlib import Path

import torch
import torch.distributed as dist
from tensordict import TensorDict
from tensordict.nn import TensorDictModule
from torch import nn
from torch.nn.parallel import DistributedDataParallel

from tdhook.latent import ActivationCaching
from tdhook.session import HookSession
from tdhook.targets import Target
from tdhook.workflow import Workflow

handoff_workflow = Workflow(
    TensorDictModule(
        torch.neg,
        in_keys=["input"],
        out_keys=[("result", "negated")],
    )
)
artifact = TensorDict(
    {
        "input": torch.ones(1, 4),
        ("result", "negated"): torch.zeros(1, 4),
    },
    batch_size=[1],
    device="cpu",
).share_memory_()

handoff_result = handoff_workflow(nn.Identity(), artifact)
assert handoff_result is artifact
assert handoff_result.is_shared()
assert torch.equal(handoff_result["result", "negated"], -torch.ones(1, 4))

The caller passes `artifact` through native multiprocessing APIs. Preallocate every declared output before `share_memory_()` or `consolidate()` locks the TensorDict structure. Inputs must be detached CPU tensors; output shape, dtype, and device must already match; and steps must retain in-place TensorDict semantics. Deferred-backward graphs cannot cross this process boundary. TDHook raises `WorkflowHandoffError` rather than silently replacing incompatible storage.

## Rank-local DDP execution

Every rank creates its own `HookSession` or calls its own `Workflow`. Target resolution starts from the underlying rank-local model, so the same path works before and after DDP wrapping—even when the underlying model itself has a first child named `module`. Captures, replacements, stops, cleanup, plans, programs, and TensorDict results are not aggregated. Keep the rank beside any result that leaves the worker.

A real job supplies its externally managed process group and one DDP replica per worker. The single-rank Gloo group below keeps this notebook deterministic while exercising the same TDHook boundary; the test suite covers two ranks.

In [ ]:
class CollisionModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.module = nn.Identity()
        self.linear = nn.Linear(2, 2, bias=False)

    def forward(self, value):
        return self.linear(self.module(value))


rendezvous_directory = tempfile.TemporaryDirectory()
rendezvous = Path(rendezvous_directory.name, "gloo-rendezvous")
dist.init_process_group(
    backend="gloo",
    init_method=f"file://{rendezvous}",
    rank=0,
    world_size=1,
)
model = CollisionModel()
with torch.no_grad():
    model.linear.weight.copy_(torch.eye(2))
ddp_model = DistributedDataParallel(model)
target = Target("module", "activation", -1, (0,))
inputs = torch.tensor([[1.0, 2.0]])

assert target.validate(ddp_model) is model.module

In [ ]:
with HookSession(ddp_model) as session:
    captured = session.capture(target)
    session.replace(target, 10)
    replaced = ddp_model(inputs)

assert torch.equal(captured.value, torch.tensor([[1.0]]))
assert torch.equal(replaced, torch.tensor([[10.0, 2.0]]))
assert not model.module._forward_hooks

workflow = Workflow(ActivationCaching(target, cache_key="activations"))
workflow_result = workflow(ddp_model, TensorDict({"input": inputs}, batch_size=[1]))
assert torch.equal(workflow_result["activations", "module"], torch.tensor([[1.0]]))
assert not model.module._forward_hooks

with HookSession(ddp_model) as session:
    stopped = session.stop("module")
    ddp_model(inputs)

rank_local_result = {"rank": dist.get_rank(), "capture": captured.value}
assert stopped.reached
assert rank_local_result["rank"] == 0
assert not model.module._forward_hooks

In [ ]:
dist.destroy_process_group()
rendezvous_directory.cleanup()

## Ownership boundary

TDHook does not own process-group initialization, job launch, worker pools, result aggregation, graph partitioning, sharding, elasticity, recovery, or FSDP. Local shared-memory handoff does not transfer a live `HookSession`, and DDP support does not imply cross-rank Workflow artifact transport.